

*   The code automatically infers the class label for each image based on the folder it is stored in.
*   It standardizes lesion attributes (color, border shape, surface texture, location) into consistent categories.
*   It then generates a synthetic clinical symptom note using controlled medical vocabulary to resemble dermatologist descriptions.
*   These generated notes help simulate real clinical metadata for each image.
*   The final output is organized and saved into a CSV file, containing:
    * Image ID / Path
    * Class Label
    * Standardized Lesion Attributes
    * Synthetic Symptom Note
* This CSV will be used in the multimodal model, where the model learns from both images and clinical text to improve diagnostic performance.



# **Importing Libraries**

In [ ]:
import os, csv, re
import random
import cv2, numpy as np
from math import pi
from skimage import measure
from skimage.feature import local_binary_pattern
from scipy.signal import savgol_filter

# **CSV Columns**

In [ ]:
CSV_COLUMNS = [
    "image_id",
    "label",
    "localization",
    "Color Descriptor",
    "Surface Roughness",
    "Border Regularity",
    "Area (px^2)",
    "Perimeter (px)",
    "Size Category",
    "Redness Level",
    "Synthetic Symptom note",
]

# **Image Preprocessing and Utility Functions**

In [ ]:
def make_mask_from_black(img_bgr, thr=8):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = (gray > thr).astype(np.uint8)
    n, lab = cv2.connectedComponents(mask)
    if n > 1:
        cnts = np.bincount(lab.ravel()); cnts[0] = 0
        mask = (lab == np.argmax(cnts)).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8), 1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((3,3), np.uint8), 1)
    return mask

def gray_world_white_balance(bgr):
    bgr = bgr.astype(np.float32)
    mean_bgr = bgr.reshape(-1,3).mean(axis=0) + 1e-6
    gray = mean_bgr.mean()
    scale = gray / mean_bgr
    wb = np.clip(bgr * scale, 0, 255).astype(np.uint8)
    return wb

def shades_of_gray_wb(bgr, p=6):
    img = bgr.astype(np.float32) + 1e-6
    B, G, R = cv2.split(img)
    rb = np.mean(R**p)**(1.0/p)
    gb = np.mean(G**p)**(1.0/p)
    bb = np.mean(B**p)**(1.0/p)
    meanI = (rb + gb + bb)/3.0
    scale = (meanI/bb, meanI/gb, meanI/rb)
    out = cv2.merge([B*scale[0], G*scale[1], R*scale[2]])
    return np.clip(out, 0, 255).astype(np.uint8)

def specular_mask(bgr, v_thr=240, s_thr=25):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    V, S = hsv[...,2], hsv[...,1]
    return ((V >= v_thr) & (S <= s_thr)).astype(np.uint8)

def to_hex(rgb):
    return "#{:02X}{:02X}{:02X}".format(*np.clip(np.round(rgb),0,255).astype(int))

# **Metadata Loader and Normalization Function**

In [ ]:
def load_metadata_map(meta_csv_path):
    """
    Returns a dict: image_id -> {'location': str or None, 'meta_row': dict or None}
    """
    meta_map = {}
    if not (meta_csv_path and os.path.exists(meta_csv_path)):
        return meta_map

    def norm_val(x):
        if x is None:
            return None
        s = str(x).strip()
        return None if s == "" or s.lower() == "nan" else s

    location_keys = ["localization", "anatom_site_general", "anatom_site_general_challenge", "anatom_site"]

    with open(meta_csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row_lc = {(k or "").strip().lower(): norm_val(v) for k, v in row.items()}

            rid = row_lc.get("image_id") or row_lc.get("filename") or row_lc.get("image") or row_lc.get("name")
            if rid:
                rid = os.path.splitext(os.path.basename(rid))[0]
            if not rid:
                continue

            location = None
            for lk in location_keys:
                if row_lc.get(lk) is not None:
                    location = row_lc[lk]
                    break

            canonical_keys = [
                "mm_per_pixel", "pixel_spacing_mm", "physical_pixel_spacing_mm",
                "x_mm_per_px", "y_mm_per_px",
                "clin_size_long_diameter_mm", "lesion_diameter_mm",
                "lesion_size_long_diameter_mm", "diameter_mm", "size_mm",
            ]
            meta_row = {}
            for k in canonical_keys:
                if row_lc.get(k) is not None:
                    meta_row[k] = row_lc[k]

            if rid not in meta_map:  # keep first occurrence
                meta_map[rid] = {"location": location, "meta_row": (meta_row if meta_row else None)}

    return meta_map

# **Redness Quantification and Severity Assessment Function**

In [ ]:
def redness_metrics(orig_bgr, lesion_mask, a_abs_thr=15, rel_pct=70):
    wb = gray_world_white_balance(orig_bgr)
    spec = specular_mask(wb)
    lesion_core = (lesion_mask.astype(bool) & (~spec.astype(bool)))

    lab = cv2.cvtColor(wb, cv2.COLOR_BGR2LAB)
    A = lab[...,1].astype(np.int16)  # 0..255 with 128 as zero
    a_star = A - 128

    vals = a_star[lesion_core]
    if vals.size == 0:
        return dict(ratio=0.0, level="none",
                    mean_hex="#000000",
                    red_mask=np.zeros_like(lesion_mask, np.uint8))

    thr_rel = np.percentile(vals, rel_pct)
    thr = max(a_abs_thr, thr_rel)
    red_mask = (a_star > thr) & lesion_core
    ratio = float(red_mask.sum()) / float(lesion_core.sum())

    pct = ratio*100
    if pct < 3: level = "none"
    elif pct < 8: level = "mild"
    elif pct < 20: level = "moderate"
    else: level = "severe"

    return dict(ratio=ratio, level=level, mean_hex="#000000", red_mask=red_mask.astype(np.uint8))

# **Surface Roughness + Border Regularity**

In [ ]:
def ensure_odd(n):
    n = int(n); return n if n % 2 == 1 else n + 1

def lesion_window_size(mask, frac=0.05, mi=7, ma=41):
    ys, xs = np.where(mask > 0)
    if ys.size == 0: return mi
    d = max(ys.max()-ys.min()+1, xs.max()-xs.min()+1)
    return ensure_odd(int(np.clip(d * frac, mi, ma)))

def crofton_perimeter(mask_bool):
    return float(measure.perimeter_crofton(mask_bool.astype(bool), directions=4))

def resample_contour_uniform(contour, n=512, smooth_win=21, poly=3):
    pts = contour.squeeze().astype(np.float64)
    if pts.ndim != 2 or pts.shape[0] < 5: return pts
    if np.all(pts[0] == pts[-1]): pts = pts[:-1]
    d = np.sqrt(np.sum(np.diff(pts, axis=0)**2, axis=1))
    s = np.hstack(([0.0], np.cumsum(d)))
    if s[-1] == 0: return pts
    t = np.linspace(0, s[-1], n)
    x = np.interp(t, s, pts[:,0]); y = np.interp(t, s, pts[:,1])
    win = min(smooth_win, n - (1 - n % 2)); win = max(win, 5)
    if win % 2 == 0: win += 1
    x = savgol_filter(x, win, poly, mode='interp')
    y = savgol_filter(y, win, poly, mode='interp')
    return np.column_stack([x, y])

def curvature_from_polyline(pts):
    dx = np.gradient(pts[:,0]); dy = np.gradient(pts[:,1])
    ddx = np.gradient(dx);       ddy = np.gradient(dy)
    denom = (dx*dx + dy*dy)**1.5 + 1e-9
    return np.abs(dx*ddy - dy*ddx) / denom

def remove_specular(img_bgr, mask, v_thr=0.97, s_thr=0.10):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    S = hsv[:,:,1]/255.0; V = hsv[:,:,2]/255.0
    spec = (V > v_thr) & (S < s_thr)
    valid = (mask > 0) & (~spec)
    return valid.astype(np.uint8), spec.astype(np.uint8)

def local_std_map(gray, ksize):
    g = gray.astype(np.float32)
    m1 = cv2.blur(g, (ksize, ksize))
    m2 = cv2.blur(g*g, (ksize, ksize))
    return np.sqrt(np.clip(m2 - m1*m1, 0, None))

def compute_border_metrics(mask):
    A = float(mask.sum())
    P = crofton_perimeter(mask)
    circularity = (4*pi*A)/(P**2 + 1e-9) if P>0 else 0.0

    cnts, _ = cv2.findContours((mask*255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cnts, key=cv2.contourArea) if cnts else None
    if c is None: return None

    hull = cv2.convexHull(c)
    A_convex = float(cv2.contourArea(hull)) + 1e-9
    solidity = float(A / A_convex)

    M = cv2.moments(c); cx, cy = M['m10']/M['m00'], M['m01']/M['m00']
    pts = c[:,0,:]
    r = np.sqrt((pts[:,0]-cx)**2 + (pts[:,1]-cy)**2)
    radial_var = float(np.var(r)/(np.mean(r)**2 + 1e-9))

    cs = resample_contour_uniform(c, n=512, smooth_win=21, poly=3)
    kappa = curvature_from_polyline(cs)
    curv_mean, curv_std = float(np.mean(kappa)), float(np.std(kappa))

    z = cs[:,0] + 1j*cs[:,1]; z -= z.mean()
    F = np.fft.fft(z); mag = np.abs(F); mag = mag/(mag[1] + 1e-9)
    hf_ratio = float(np.sum(mag[8:]**2) / (np.sum(mag**2) + 1e-9))

    BII = 0.5*(1 - circularity) + 0.3*(1 - solidity) + 0.2*np.clip(radial_var, 0, 1)

    return {"area_px":A, "perimeter_crofton":P, "circularity":circularity,
            "solidity":solidity, "radial_variance":radial_var,
            "curvature_mean":curv_mean, "curvature_std":curv_std,
            "fourier_hf_ratio":hf_ratio, "BII":float(BII),
            "contour":c, "contour_smooth":cs}

def compute_surface_metrics(img_bgr, mask):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[:,:,0]

    k = lesion_window_size(mask, frac=0.05, mi=7, ma=41)
    std_map = local_std_map(L, k)

    gx = cv2.Sobel(L, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(L, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)

    valid_mask, spec_mask = remove_specular(img_bgr, mask)
    m = valid_mask.astype(bool)
    std_vals, grad_vals = std_map[m], grad_mag[m]

    std_p90  = float(np.percentile(std_vals, 90)) if std_vals.size else 0.0
    grad_p90 = float(np.percentile(grad_vals, 90)) if grad_vals.size else 0.0

    def lbp_entropy(gray, mask, P, R):
        lbp = local_binary_pattern(gray, P, R, method='uniform')
        vals = lbp[mask.astype(bool)]
        n_bins = P + 2
        hist, _ = np.histogram(vals, bins=np.arange(n_bins+1), density=True)
        hist = hist + 1e-12
        return float(-np.sum(hist*np.log2(hist)))

    lbp_ent = float((lbp_entropy(L, valid_mask, 8, 1) + lbp_entropy(L, valid_mask, 16, 2))/2.0)

    SRI = 0.5*(std_p90/20.0) + 0.3*(grad_p90/40.0) + 0.2*((lbp_ent-1.5)/1.5)

    return {"std_p90":std_p90, "grad_p90":grad_p90, "lbp_entropy_ms":lbp_ent,
            "SRI":float(SRI),
            "specular_fraction":float(np.mean(spec_mask[mask>0]))}

def classify_border(border):
    if border is None:
        return "unspecified", 0.0
    bii = border["BII"]; circ = border["circularity"]; sol = border["solidity"]; rvar = border["radial_variance"]
    if (bii < 0.10) and (circ >= 0.85) and (rvar <= 0.015) and (sol >= 0.93):
        label = "regular"
    elif (bii >= 0.22) or (circ < 0.80) or (rvar > 0.03) or (sol < 0.90):
        label = "irregular"
    else:
        label = "mildly_irregular"
    conf = float(np.clip(1.0 - bii/0.35, 0, 1))
    return label, conf

def classify_surface(surface):
    SRI  = surface["SRI"]; lbp = surface["lbp_entropy_ms"]; grad = surface["grad_p90"]; spec = surface["specular_fraction"]
    if (SRI >= 3.5) or (spec >= 0.005):
        label = "keratotic"
    elif (SRI >= 2.0) or (lbp >= 3.2):
        label = "rough"
    elif (SRI >= 1.0):
        label = "mildly_rough"
    else:
        label = "smooth" if (grad < 60 and lbp < 2.6) else "mildly_rough"
    edges = np.array([0.0, 1.0, 2.0, 3.5], dtype=float)
    dmin = float(np.min(np.abs(SRI - edges)))
    conf = float(np.clip(dmin/1.5, 0, 1))
    return label, conf

# **Color Descriptor**

In [ ]:
def compute_color_descriptor_robust(orig_bgr, mask,
                                    gray_C_thr=9.0, gray_ab_thr=8.0,
                                    pink_a_thr=16.0, pink_L_thr=68.0,
                                    red_a_thr=22.0,
                                    brown_a_thr=6.0, brown_b_thr=12.0,
                                    light_dark_L_cut=55.0,
                                    min_share=0.35, min_gap=0.07):
    wb = shades_of_gray_wb(orig_bgr, p=6)
    hsv = cv2.cvtColor(wb, cv2.COLOR_BGR2HSV)
    spec = ((hsv[...,2] > 240) & (hsv[...,1] < 25))
    lesion_core = (mask.astype(bool) & (~spec))

    if not np.any(lesion_core):
        return dict(label="unknown", dominant_hex="#000000", dominant_share=0.0,
                    shares_by_class={}, used_pixels=0)

    lab = cv2.cvtColor(wb, cv2.COLOR_BGR2LAB)
    Lcv, Acv, Bcv = lab[...,0], lab[...,1], lab[...,2]

    ys, xs = np.where(lesion_core)
    Ls  = Lcv[ys, xs].astype(np.float32) * (100.0/255.0)
    a_s = Acv[ys, xs].astype(np.float32) - 128.0
    b_s = Bcv[ys, xs].astype(np.float32) - 128.0
    C   = np.hypot(a_s, b_s)
    total = float(Ls.size)

    black  = (Ls < 22)
    gray   = (~black) & (C < gray_C_thr) & (np.abs(a_s) < gray_ab_thr) & (np.abs(b_s) < gray_ab_thr) & (35 <= Ls) & (Ls <= 85)
    pink   = (~black) & (~gray) & (Ls >= pink_L_thr) & (a_s >= pink_a_thr)
    reddish= (~black) & (~gray) & (~pink) & (a_s >= red_a_thr)
    brownish = (~black) & (~gray) & (~pink) & (~reddish) & (a_s >= brown_a_thr) & (b_s >= brown_b_thr)
    light_brown = brownish & (Ls >= light_dark_L_cut)
    dark_brown  = brownish & (Ls <  light_dark_L_cut)

    assigned = black | gray | pink | reddish | light_brown | dark_brown
    rest = ~assigned
    light_brown |= (rest & (Ls >= light_dark_L_cut))
    dark_brown  |= (rest & (Ls <  light_dark_L_cut))

    shares = {
        "light_brown":  float(np.count_nonzero(light_brown))/total,
        "dark_brown":   float(np.count_nonzero(dark_brown))/total,
        "reddish":      float(np.count_nonzero(reddish))/total,
        "pink":         float(np.count_nonzero(pink))/total,
        "gray":         float(np.count_nonzero(gray))/total,
        "black":        float(np.count_nonzero(black))/total,
    }
    ordered = sorted(shares.items(), key=lambda kv: kv[1], reverse=True)
    label, s1 = ordered[0]
    s2 = ordered[1][1] if len(ordered) > 1 else 0.0
    if (s1 < min_share) or ((s1 - s2) < min_gap):
        label = "mixed"

    lesion_rgb = orig_bgr[..., ::-1]
    if label == "mixed":
        full_sel = mask.astype(bool)
    else:
        sel = {
            "light_brown": light_brown, "dark_brown": dark_brown,
            "reddish": reddish, "pink": pink, "gray": gray, "black": black
        }[label]
        full_sel = np.zeros(mask.shape, dtype=bool)
        full_sel[ys, xs] = sel

    if not np.any(full_sel):
        full_sel = mask.astype(bool)

    mean_rgb = lesion_rgb[full_sel].mean(axis=0)
    hex_color = to_hex(mean_rgb)

    return dict(label=label, dominant_hex=hex_color, dominant_share=float(s1),
                shares_by_class=shares, used_pixels=int(total))

# **Size Category**

In [ ]:
def get_region_axes_px(mask):
    lab = measure.label(mask.astype(bool))
    props = measure.regionprops(lab)
    if not props:
        return 0.0, 0.0
    r = max(props, key=lambda p: p.area)
    return float(r.major_axis_length), float(r.minor_axis_length)

def effective_diameter_px(area_px):
    return 2.0 * np.sqrt(float(area_px) / pi)

def try_extract_mm_per_pixel_from_meta(meta_row):
    if meta_row is None:
        return None, "none"
    for k in ["mm_per_pixel", "pixel_spacing_mm", "physical_pixel_spacing_mm",
              "x_mm_per_px", "y_mm_per_px"]:
        if k in meta_row and meta_row[k] not in (None, "", "nan"):
            val = meta_row[k]
            try:
                if isinstance(val, str) and ";" in val:
                    xs, ys = val.split(";")
                    xs, ys = float(xs), float(ys)
                    return float((xs + ys)/2.0), f"meta:{k}"
                return float(val), f"meta:{k}"
            except Exception:
                pass
    return None, "none"

def try_extract_long_diameter_mm(meta_row):
    if meta_row is None:
        return None, "none"
    for k in ["clin_size_long_diameter_mm", "lesion_diameter_mm",
              "lesion_size_long_diameter_mm", "diameter_mm", "size_mm"]:
        if k in meta_row and meta_row[k] not in (None, "", "nan"):
            try:
                return float(meta_row[k]), k
            except Exception:
                pass
    return None, "none"

def size_category_from_mm(d_mm):
    if d_mm is None: return None
    if d_mm < 5.0: return "small"
    if d_mm <= 10.0: return "medium"
    return "large"

def size_category_from_relative(area_ratio, d_eff_px):
    if area_ratio < 0.04: return "small"
    if area_ratio <= 0.12: return "medium"
    return "large"

def compute_size_category(img_bgr, mask, meta_row=None):
    H, W = mask.shape
    area_px = int(mask.sum())
    if area_px == 0:
        return dict(method="none", area_px=0, area_ratio=0.0, eff_diam_px=0.0,
                    major_axis_px=0.0, minor_axis_px=0.0,
                    mm_per_px=None, diameter_mm=None, category=None)

    eff_px = effective_diameter_px(area_px)
    major_px, minor_px = get_region_axes_px(mask)
    area_ratio = area_px / float(H*W)

    mm_per_px, mpp_src = try_extract_mm_per_pixel_from_meta(meta_row)
    diam_mm_meta, diam_key = try_extract_long_diameter_mm(meta_row)
    infer_used = False
    if mm_per_px is None and diam_mm_meta is not None and major_px > 0:
        mm_per_px = diam_mm_meta / major_px
        infer_used = True

    d_mm = None
    method = "relative"
    if mm_per_px is not None:
        d_mm = eff_px * mm_per_px
        method = f"metric({mpp_src})" if not infer_used else f"inferred({diam_key}/major_axis)"
        cat = size_category_from_mm(d_mm)
    else:
        cat = size_category_from_relative(area_ratio, eff_px)

    return dict(
        method=method, area_px=int(area_px), area_ratio=float(area_ratio),
        eff_diam_px=float(eff_px), major_axis_px=float(major_px), minor_axis_px=float(minor_px),
        mm_per_px=(None if mm_per_px is None else float(mm_per_px)),
        diameter_mm=(None if d_mm is None else float(d_mm)), category=cat
    )

# **Clinical Description Language Generator for Lesion Reports (Template-Based NLG with Lexical Split Control)**

In [ ]:
def a_or_an(phrase):
    first = phrase.strip().lower()[0] if phrase and phrase.strip() else "a"
    return "an" if first in "aeiou" else "a"

def maybe(prob):
    return random.random() < prob

def inject_typos(text, prob=0.12, max_edits=2):
    if not maybe(prob):
        return text
    keyboard_neighbors = {
        'a':'qwsz', 'b':'vghn', 'c':'xdfv', 'd':'erfcxs', 'e':'wsdr',
        'f':'rtgdvc', 'g':'tyfhvb', 'h':'yugjbn', 'i':'ujko', 'j':'uikhmn',
        'k':'ijolm', 'l':'kop', 'm':'njk', 'n':'bhjm', 'o':'iklp',
        'p':'ol', 'q':'w', 'r':'edft', 's':'awedxz', 't':'rfgy',
        'u':'yhji', 'v':'cfgb', 'w':'qe', 'x':'zsdc', 'y':'tugh', 'z':'asx'
    }
    s = list(text)
    edits = 0
    for _ in range(len(s)):
        if edits >= max_edits:
            break
        i = random.randrange(0, len(s))
        ch = s[i].lower()
        op = random.choice(["del","swap","sub"])
        if op == "del" and s[i].isalnum():
            s.pop(i); edits += 1
        elif op == "swap" and i < len(s)-1 and s[i].isalnum() and s[i+1].isalnum():
            s[i], s[i+1] = s[i+1], s[i]; edits += 1
        elif op == "sub" and ch in keyboard_neighbors and s[i].isalnum():
            s[i] = random.choice(keyboard_neighbors[ch]); edits += 1
    return "".join(s)

def drop_stopwords(text, prob=0.12):
    if not maybe(prob):
        return text
    stops = {"the","a","an","is","are"}
    tokens = text.split()
    kept = []
    for t in tokens:
        if t.lower() in stops and maybe(0.5):
            continue
        kept.append(t)
    return " ".join(kept)

def punct_variants(text, prob=0.2):
    if not maybe(prob):
        return text
    text = re.sub(
        r", (with|showing|featuring|displaying|exhibiting|accompanied by) ",
        lambda m: random.choice([", " + m.group(1) + " ", "; " + m.group(1) + " ", " — " + m.group(1) + " "]),
        text, count=1
    )
    return text

# Canonical label -> list of synonyms (keys match your classifiers)
COLOR_SYNS = {
    "gray":        ["gray", "grey", "slate", "ashen"],
    "dark_brown":  ["dark brown", "dark-brown", "mahogany"],
    "light_brown": ["light brown", "light-brown", "tan", "fawn"],
    "black":       ["black", "jet-black", "charcoal"],
    "pink":        ["pink", "rosy", "rose-hued"],
    "reddish":     ["reddish", "red-toned", "brick-red"],
    "mixed":       ["variegated", "mixed coloration", "multicolored"],
    # "unknown" will be treated as missing (no color phrase)
}

SURFACE_SYNS = {
    "keratotic":     ["keratotic", "hyperkeratotic", "crusted"],
    "rough":         ["rough", "coarse", "scaly", "uneven", "gritty"],
    "mildly_rough":  ["mildly rough", "slightly rough", "finely scaly"],
    "smooth":        ["smooth", "even", "polished", "velvety"],
}

BORDER_SYNS = {
    "regular":           ["regular", "smooth", "well circumscribed", "well-circumscribed"],
    "mildly_irregular":  ["mildly irregular", "slightly irregular"],
    "irregular":         ["irregular", "ill-defined", "notched"],
}

REDNESS_SYNS = {
    "none":     ["without visible redness", "no erythema", "no redness", "non-erythematous"],
    "mild":     ["mild redness", "mild erythema", "slight redness"],
    "moderate": ["moderate redness", "moderate erythema"],
    "severe":   ["marked redness", "severe erythema", "pronounced redness"],
}

WITH_WORDS = ["with", "showing", "featuring", "displaying", "exhibiting", "accompanied by", "demonstrating"]
LESION_NOUNS = ["lesion", "skin lesion", "plaque", "patch", "macule", "papule"]  # keep usage label-agnostic

LOCATION_SYNS = {
    "lower_extremity": ["lower extremity", "leg", "lower limb"],
    "upper_extremity": ["upper extremity", "arm", "upper limb"],
    "neck":            ["neck", "cervical region"],
    "unspecified":     ["unspecified site", "unspecified location", "unknown site"],
    # add more if needed: "face": [...], "trunk": [...], etc.
}

OPENERS = [
    "There is {art} {desc} {lesion} on the {loc}",
    "On the {loc}, {art} {desc} {lesion} is {present}",
    "{Lesion_cap} on the {loc} {copula} {desc}",
    "Examination of the {loc} {reveals} {art} {desc} {lesion}",
    "Inspection of the {loc} {reveals} {art} {desc} {lesion}",
    "At the {loc}, we {observe} {art} {desc} {lesion}",
    "The {loc} {shows} {art} {desc} {lesion}",
    "Located on the {loc} is {art} {desc} {lesion}",
    "In the region of the {loc}, there is {art} {desc} {lesion}",
    "Findings at the {loc}: {Art} {desc} {lesion}",
    "{Art} {desc} {lesion} is identified on the {loc}",
    "{Art} {desc} {lesion} is appreciated on the {loc}",
    "Derm exam: {art} {desc} {lesion} on the {loc}",
    "We note {art} {desc} {lesion} on the {loc}",
    "Observed at the {loc} is {art} {desc} {lesion}",
]
PRESENT_SYNS = ["present", "noted", "observed", "evident", "apparent", "seen", "identified"]
REVEALS_SYNS = ["reveals", "shows", "demonstrates", "exhibits", "highlights"]
SHOWS_SYNS   = ["shows", "demonstrates", "exhibits", "displays"]
COPULAE      = ["is", "appears", "looks"]

def make_split_sets(full_list, holdout_n=1):
    if holdout_n <= 0 or holdout_n >= len(full_list):
        return set(full_list), set()
    items = full_list[:]
    random.shuffle(items)
    held = set(items[:holdout_n])
    train = set(items[holdout_n:])
    return train, held

def build_split_lexicons(holdout_per_value=1, holdout_templates=5, seed=13):
    random.seed(seed)

    def split_attr(attr_map):
        train_dict, test_dict = {}, {}
        for k, syns in attr_map.items():
            syns = list(syns)
            tr, ho = make_split_sets(syns, holdout_n=min(holdout_per_value, max(0, len(syns)-1)))
            train_dict[k] = tr or set(syns)
            test_dict[k] = ho
        return {"train":train_dict, "lex_test":test_dict, "current":{}}

    split_objs = {
        "color":     split_attr(COLOR_SYNS),
        "surface":   split_attr(SURFACE_SYNS),
        "border":    split_attr(BORDER_SYNS),
        "redness":   split_attr(REDNESS_SYNS),
        "location":  split_attr(LOCATION_SYNS),
        "with":      {"train": set(WITH_WORDS[:-2]), "lex_test": set(WITH_WORDS[-2:]), "current": set()},
        "lesion":    {"train": set(LESION_NOUNS[:-2]), "lex_test": set(LESION_NOUNS[-2:]), "current": set()},
        "templates": {"train": set(), "lex_test": set(), "current": set()},
    }
    all_t_ids = set(range(len(OPENERS)))
    tr_t, ho_t = make_split_sets(list(all_t_ids), holdout_templates)
    split_objs["templates"]["train"]    = tr_t
    split_objs["templates"]["lex_test"] = ho_t
    return split_objs

def set_split_mode(split_objs, mode="train"):
    assert mode in ("train","lex_test")
    for k, v in split_objs.items():
        if k in ("color","surface","border","redness","location"):
            v["current"] = {key: (v[mode].get(key) or v["train"].get(key)) for key in v["train"].keys()}
        elif k in ("with","lesion","templates"):
            v["current"] = (v[mode] or v["train"])
    return split_objs

def pick_syn(split_syns, key, fallback_text=None):
    # Robust picker with fallback
    cur = split_syns["current"]
    if key in cur and len(cur[key]) > 0:
        return random.choice(list(cur[key]))
    return fallback_text if fallback_text else (key.replace("_", " ") if key else "")

def border_phrase(border_label, split_objs):
    b_syn = pick_syn(split_objs["border"], border_label)
    return f"{b_syn} borders"

def redness_phrase(red_label, split_objs):
    return pick_syn(split_objs["redness"], red_label)

def size_phrase(size_cat):
    if not size_cat: return None
    forms = [
        f"{size_cat.lower()} in size",
        f"{size_cat.lower()}-sized",
        f"of {size_cat.lower()} size",
        f"measuring {size_cat.lower()}",
    ]
    return random.choice(forms)

def desc_phrase(color_label, surface_label, split_objs):
    parts = []
    if color_label:
        parts.append(pick_syn(split_objs["color"], color_label))
    if surface_label:
        parts.append(pick_syn(split_objs["surface"], surface_label))
    if len(parts) == 2 and maybe(0.5):
        parts = parts[::-1]
    joiner = ", " if maybe(0.6) else " "
    return joiner.join(parts) if parts else "nonspecific"

def loc_phrase(location_label, split_objs):
    return pick_syn(split_objs["location"], location_label, fallback_text="unspecified site")

def with_word(split_objs):
    return random.choice(list(split_objs["with"]["current"]))

def lesion_noun(split_objs):
    return random.choice(list(split_objs["lesion"]["current"]))

def compose_note_v2(
    location_label,
    color_label,
    surface_label,
    border_label,
    size_cat,
    redness_label,
    *,
    split_objs,
    include_measurements=False,
    area_px2=None,
    perimeter_px=None,
    noise=True
):
    loc = loc_phrase(location_label, split_objs)
    desc = desc_phrase(color_label, surface_label, split_objs)
    lesion = lesion_noun(split_objs)
    art = a_or_an(desc)
    Art = art.capitalize()

    segs = []
    if border_label and border_label != "unspecified":
        segs.append(border_phrase(border_label, split_objs))
    if size_cat:
        segs.append(size_phrase(size_cat))
    if redness_label:
        segs.append(redness_phrase(redness_label, split_objs))
    if include_measurements and (area_px2 or perimeter_px):
        m_parts = []
        if area_px2:     m_parts.append(f"~{int(round(area_px2))} px² area")
        if perimeter_px: m_parts.append(f"perimeter ~{round(float(perimeter_px), 1)} px")
        if m_parts:
            segs.append(", ".join(m_parts))

    random.shuffle(segs)
    attr_clause = f", {with_word(split_objs)} " + " and ".join(segs) if segs else ""

    t_ids = split_objs["templates"]["current"]
    t_id = random.choice(list(t_ids))
    opener = OPENERS[t_id].format(
        art=art,
        Art=Art,
        desc=desc,
        lesion=lesion,
        Lesion_cap=lesion.capitalize(),
        loc=loc,
        present=random.choice(PRESENT_SYNS),
        copula=random.choice(COPULAE),
        reveals=random.choice(REVEALS_SYNS),
        observe=random.choice(["observe", "note, and", "identify"]),
        shows=random.choice(SHOWS_SYNS),
    )

    sentence = opener + attr_clause + "."
    if noise:
        sentence = punct_variants(sentence, prob=0.25)
        sentence = drop_stopwords(sentence, prob=0.08)
        sentence = inject_typos(sentence, prob=0.08, max_edits=2)
    sentence = re.sub(r"\s{2,}", " ", sentence)
    sentence = sentence.replace(" ,", ",").replace(" .", ".")
    return sentence


# **Canonicalization Functions for Standardizing Lesion Attribute Keys**

In [ ]:
def canonicalize_location_key(loc_raw):
    if not loc_raw:
        return "unspecified"
    s = str(loc_raw).strip().lower().replace("-", " ")
    if any(k in s for k in ["lower extremity","leg","calf","thigh","lower limb","shin","knee","ankle","foot"]):
        return "lower_extremity"
    if any(k in s for k in ["upper extremity","arm","forearm","upper limb","elbow","wrist","hand"]):
        return "upper_extremity"
    if any(k in s for k in ["neck","cervical"]):
        return "neck"
    return "unspecified"

def canonicalize_color_key(label):
    if not label or label in ("unknown", ""):
        return None
    return str(label).strip().lower()

def canonicalize_border_key(label):
    if not label or label == "unspecified":
        return None
    return str(label).strip().lower()

def canonicalize_surface_key(label):
    if not label:
        return None
    return str(label).strip().lower()

# **Label Inference from Directory Structure**

In [ ]:
def infer_label_from_path(image_path, root):
    # Returns the first folder under root as the class label.
    # Example: root/.../test/NV/img.png -> "NV"
    rel = os.path.relpath(image_path, root)
    parts = [p for p in rel.split(os.sep) if p not in (".", "")]
    if len(parts) >= 2:
        return parts[0]
    # Fallback: parent dir name if image is directly under root
    parent = os.path.basename(os.path.dirname(image_path))
    return parent if parent and parent != os.path.basename(root) else "UNKNOWN"

# **Batch processing**

End-to-End Image Processing Pipeline: Feature Extraction, Clinical Note Generation, and CSV Dataset Construction

In [ ]:
def iter_images(root, extensions=(".jpg",".jpeg",".png",".bmp",".tif",".tiff"), recursive=True, limit_per_folder=None):
    if recursive:
        for dirpath, _, filenames in os.walk(root):
            imgs = [f for f in sorted(filenames) if f.lower().endswith(extensions)]
            if limit_per_folder is not None:
                imgs = imgs[:int(limit_per_folder)]
            for fn in imgs:
                full = os.path.join(dirpath, fn)
                label = infer_label_from_path(full, root)  # NEW
                yield full, label
    else:
        filenames = [f for f in sorted(os.listdir(root)) if os.path.isfile(os.path.join(root, f))]
        imgs = [f for f in filenames if f.lower().endswith(extensions)]
        if limit_per_folder is not None:
            imgs = imgs[:int(limit_per_folder)]
        for fn in imgs:
            full = os.path.join(root, fn)
            label = infer_label_from_path(full, root)  # NEW
            yield full, label

def analyze_image_to_note(image_path, meta_map, gen_ctx):
    """
    gen_ctx: dict(
      split_objs,
      include_measurements: bool,
      noise: bool
    )
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    meta_entry = meta_map.get(image_id, {}) if meta_map else {}
    loc_raw = meta_entry.get("location")
    meta_row = meta_entry.get("meta_row")

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise RuntimeError(f"Failed to read image: {image_path}")

    mask = make_mask_from_black(img_bgr)

    # Components
    redness_out = redness_metrics(img_bgr, mask)
    surface = compute_surface_metrics(img_bgr, mask)
    surface_label, _ = classify_surface(surface)

    border = compute_border_metrics(mask)
    border_label, _ = classify_border(border)

    color_desc = compute_color_descriptor_robust(img_bgr, mask)
    size_info = compute_size_category(img_bgr, mask, meta_row)

    # Prepare values for CSV
    if border is not None:
        area_px_csv  = int(round(border.get("area_px", 0.0)))
        perim_px_csv = float(border.get("perimeter_crofton", 0.0))
    else:
        area_px_csv  = int(mask.sum())
        perim_px_csv = 0.0

    # Canonicalize keys for the generator
    loc_key     = canonicalize_location_key(loc_raw)
    color_key   = canonicalize_color_key(color_desc["label"])
    surface_key = canonicalize_surface_key(surface_label)
    border_key  = canonicalize_border_key(border_label)
    size_cat    = (size_info.get("category") or "")
    redness_key = (redness_out.get("level") or "").strip().lower()

    note = compose_note_v2(
        location_label=loc_key,
        color_label=color_key,
        surface_label=surface_key,
        border_label=border_key,
        size_cat=size_cat,
        redness_label=redness_key,
        split_objs=gen_ctx["split_objs"],
        include_measurements=gen_ctx.get("include_measurements", False),
        area_px2=area_px_csv,
        perimeter_px=perim_px_csv,
        noise=gen_ctx.get("noise", True)
    )

    # Friendly labels for CSV (keep original text-ish forms)
    color_label_csv   = (color_desc["label"] or "").replace("_", " ").strip()
    surface_label_csv = (surface_label or "").replace("_", " ").strip()
    border_label_csv  = (border_label or "").replace("_", " ").strip()
    size_cat_csv      = (size_info.get("category") or "").strip()
    redness_level_csv = (redness_out.get("level") or "").strip()

    return {
        "image_id": image_id,
        "localization": loc_raw or "",
        "Color Descriptor": color_label_csv,
        "Surface Roughness": surface_label_csv,
        "Border Regularity": border_label_csv,
        "Area (px^2)": area_px_csv,
        "Perimeter (px)": perim_px_csv,
        "Size Category": size_cat_csv,
        "Redness Level": redness_level_csv,
        "Synthetic Symptom note": note
    }

def process_images_to_metadata(
    input_root,
    output_csv,
    metadata_csv=None,
    limit_per_folder=None,
    recursive=True,
    verbose=True,
    # NEW: generator controls
    split_mode="train",            # "train" or "lex_test"
    holdout_per_value=1,           # how many synonyms per label to hold out to lex_test
    holdout_templates=5,           # how many templates to hold out to lex_test
    split_seed=42,                 # seed for lexical split
    include_measurements=False,    # verbalize area/perimeter
    noise=True                     # inject typos/punct/stopword noise
):
    # Build lexical split pools once
    split_objs = build_split_lexicons(
        holdout_per_value=holdout_per_value,
        holdout_templates=holdout_templates,
        seed=split_seed
    )
    split_objs = set_split_mode(split_objs, mode=split_mode)

    gen_ctx = dict(
        split_objs=split_objs,
        include_measurements=include_measurements,
        noise=noise
    )

    meta_map = load_metadata_map(metadata_csv) if metadata_csv else {}
    if verbose:
        print(f"Loaded metadata rows: {len(meta_map)} | split_mode={split_mode} | seed={split_seed}")

    rows = []
    total = 0
    errors = 0

    for img_path, class_label in iter_images(input_root, recursive=recursive, limit_per_folder=limit_per_folder):
        total += 1
        try:
            row = analyze_image_to_note(img_path, meta_map, gen_ctx)
            row["label"] = class_label  # NEW: attach class label
            rows.append(row)
            if verbose and total % 25 == 0:
                print(f"Processed {total} images...")
        except Exception as e:
            errors += 1
            if verbose:
                print(f"[Error] {img_path}: {e}")

    # Write CSV
    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(CSV_COLUMNS)
        for r in rows:
            writer.writerow([
                r.get("image_id", ""),
                r.get("label", ""),  # NEW
                r.get("localization", ""),
                r.get("Color Descriptor", ""),
                r.get("Surface Roughness", ""),
                r.get("Border Regularity", ""),
                r.get("Area (px^2)", ""),
                r.get("Perimeter (px)", ""),
                r.get("Size Category", ""),
                r.get("Redness Level", ""),
                r.get("Synthetic Symptom note", ""),
            ])

    if verbose:
        print(f"Done. Wrote {len(rows)} rows to {output_csv}. Total processed: {total}. Errors: {errors}.")

# **Main**

In [ ]:
if __name__ == "__main__":
    # Configure these paths
    input_root = "output-2/test"           # Folder containing images (recursively)
    metadata_csv = "HAM10000_metadata.csv" # Metadata file to read localization (optional)
    output_csv = "symptom_metadata_test(N).csv"  # Output CSV path

    # Limit how many images per folder to process (set None to process all)
    limit_per_folder = None  # e.g., 100

    # For reproducibility of lexical splits and noise sampling:
    random.seed(42)

    # Process
    process_images_to_metadata(
        input_root=input_root,
        output_csv=output_csv,
        metadata_csv=metadata_csv,
        limit_per_folder=limit_per_folder,
        recursive=True,
        verbose=True,
        split_mode="train",            # change to "lex_test" to generate held-out-synonym text
        holdout_per_value=1,
        holdout_templates=5,
        split_seed=42,
        include_measurements=True,     # True to include area/perimeter phrases
        noise=True
    )

    # Example: Generate a lexical-test CSV by switching split_mode and output_csv
    # process_images_to_metadata(
    #     input_root=input_root,
    #     output_csv="symptom_metadata-lexTEST.csv",
    #     metadata_csv=metadata_csv,
    #     split_mode="lex_test",
    #     holdout_per_value=1,
    #     holdout_templates=5,
    #     split_seed=42,
    #     include_measurements=False,
    #     noise=True
    # )